# 🎬 Chrome Extension 비디오 생성기 (간단 버전)\n\n**한 번의 클릭으로 비디오 생성**\n\n---

## ⚡ 빠른 실행 (이 셀만 실행하면 됩니다)

In [ ]:
# 🚀 전체 자동 실행 (이 셀 하나만 실행하세요!)

!pip install -q moviepy pillow numpy opencv-python-headless imageio imageio-ffmpeg
!apt-get update && apt-get install -y ffmpeg > /dev/null 2>&1

import os
import json
import base64
from io import BytesIO
import numpy as np
from PIL import Image
from moviepy.editor import *
from google.colab import files
from IPython.display import Video, display

print("🎬 Chrome Extension 비디오 생성기")
print("="*50)

# JSON 파일 업로드
print("\n📤 Chrome Extension JSON 파일을 업로드하세요:")
uploaded = files.upload()

if uploaded:
    # JSON 로드
    json_file = list(uploaded.keys())[0]
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    session_id = data.get('session_id', 'video')
    print(f"\n✅ {data.get('image_count', 0)}개 이미지 로드")
    
    # 이미지 처리
    clips = []
    for i, img_data in enumerate(data.get('images', [])):
        if img_data['url'].startswith('data:'):
            # Data URL을 이미지로 변환
            header, encoded = img_data['url'].split(',', 1)
            img_bytes = base64.b64decode(encoded)
            img = Image.open(BytesIO(img_bytes))
            
            # 임시 파일로 저장
            temp_path = f'/tmp/img_{i}.png'
            img.save(temp_path)
            
            # 비디오 클립 생성 (3초씩)
            clip = ImageClip(temp_path, duration=3)
            clip = clip.resize((1920, 1080))  # Full HD
            clips.append(clip)
            print(f"✓ 이미지 {i+1}/{len(data['images'])} 처리")
    
    # 비디오 생성
    print("\n🎬 비디오 생성 중...")
    if len(clips) > 1:
        # 크로스페이드 전환 효과
        final_clips = [clips[0]]
        for i in range(1, len(clips)):
            clips[i] = clips[i].crossfadein(0.5)
            clips[i] = clips[i].set_start(final_clips[-1].duration - 0.5)
            final_clips.append(clips[i])
        video = CompositeVideoClip(final_clips)
    else:
        video = clips[0] if clips else None
    
    if video:
        # 비디오 저장
        output = f'{session_id}_video.mp4'
        video.write_videofile(output, fps=30, codec='libx264', audio=False, verbose=False, logger=None)
        
        print(f"\n✅ 비디오 생성 완료!")
        print(f"📹 길이: {video.duration}초")
        print(f"📁 파일: {output}")
        
        # 미리보기
        display(Video(output, embed=True, width=720))
        
        # 다운로드
        print("\n💾 다운로드 중...")
        files.download(output)
else:
    print("❌ 파일이 업로드되지 않았습니다")

---\n\n## 🎨 고급 설정 (선택사항)

In [ ]:
# 🎨 커스텀 설정으로 비디오 생성

# 설정 변경
CONFIG = {
    'duration_per_image': 2.0,    # 이미지당 시간 (초)
    'resolution': (1920, 1080),   # 해상도
    'transition': 0.5,            # 전환 효과 시간
    'fps': 30,                    # 프레임레이트
    'quality': 'high'             # 품질 (low/medium/high/ultra)
}

print("📤 Chrome Extension JSON 파일을 업로드하세요:")
uploaded = files.upload()

if uploaded:
    # JSON 로드
    json_file = list(uploaded.keys())[0]
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    clips = []
    for i, img_data in enumerate(data.get('images', [])):
        if img_data['url'].startswith('data:'):
            header, encoded = img_data['url'].split(',', 1)
            img_bytes = base64.b64decode(encoded)
            img = Image.open(BytesIO(img_bytes))
            
            temp_path = f'/tmp/img_{i}.png'
            img.save(temp_path)
            
            # Ken Burns 효과 (줌 인)
            clip = ImageClip(temp_path, duration=CONFIG['duration_per_image'])
            clip = clip.resize(CONFIG['resolution'])
            
            # 줌 효과
            clip = clip.resize(lambda t: 1 + 0.1*t/CONFIG['duration_per_image'])
            
            clips.append(clip)
            print(f"✓ 이미지 {i+1}/{len(data['images'])} 처리 (Ken Burns 효과)")
    
    # 비디오 생성 (전환 효과)
    print("\n🎬 고급 비디오 생성 중...")
    if clips:
        final_clips = [clips[0]]
        for i in range(1, len(clips)):
            clips[i] = clips[i].crossfadein(CONFIG['transition'])
            clips[i] = clips[i].set_start(final_clips[-1].duration - CONFIG['transition'])
            final_clips.append(clips[i])
        
        video = CompositeVideoClip(final_clips)
        
        # 색상 필터 (시네마틱)
        video = video.fx(vfx.colorx, 0.95)  # 약간 어둡게
        video = video.fx(vfx.lum_contrast, contrast=0.2)  # 대비 증가
        
        # 품질 설정
        quality_map = {
            'low': {'bitrate': '2M', 'crf': 28},
            'medium': {'bitrate': '5M', 'crf': 23},
            'high': {'bitrate': '10M', 'crf': 18},
            'ultra': {'bitrate': '20M', 'crf': 15}
        }
        
        q = quality_map[CONFIG['quality']]
        output = f'custom_video_{data.get("session_id", "output")}.mp4'
        
        video.write_videofile(
            output,
            fps=CONFIG['fps'],
            codec='libx264',
            bitrate=q['bitrate'],
            audio=False,
            verbose=False,
            logger=None
        )
        
        print(f"\n✅ 고급 비디오 생성 완료!")
        print(f"📹 길이: {video.duration}초")
        print(f"🎨 효과: Ken Burns + 시네마틱 필터")
        print(f"📁 파일: {output}")
        
        display(Video(output, embed=True, width=720))
        files.download(output)

---\n\n## 🎵 BGM 추가 버전

In [ ]:
# 🎵 BGM이 있는 비디오 생성

from moviepy.audio.AudioClip import AudioClip

print("📤 JSON 파일 업로드:")
uploaded = files.upload()

if uploaded:
    json_file = list(uploaded.keys())[0]
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # 이미지 처리
    clips = []
    for i, img_data in enumerate(data.get('images', [])):
        if img_data['url'].startswith('data:'):
            header, encoded = img_data['url'].split(',', 1)
            img_bytes = base64.b64decode(encoded)
            img = Image.open(BytesIO(img_bytes))
            temp_path = f'/tmp/img_{i}.png'
            img.save(temp_path)
            
            clip = ImageClip(temp_path, duration=3).resize((1920, 1080))
            clips.append(clip)
    
    # 비디오 생성
    if clips:
        video = concatenate_videoclips(clips, method="compose")
        
        # BGM 생성 (앰비언트 사운드)
        def make_bgm(t):
            frequencies = [110, 165, 220]  # A2, E3, A3
            signal = np.zeros_like(t)
            for freq in frequencies:
                signal += np.sin(2 * np.pi * freq * t) * 0.05
            return np.stack([signal, signal]).T if len(signal.shape) == 1 else signal
        
        audio = AudioClip(make_bgm, duration=video.duration)
        audio = audio.volumex(0.3)
        audio = audio.audio_fadein(2).audio_fadeout(2)
        
        video = video.set_audio(audio)
        
        output = f'video_with_bgm_{data.get("session_id", "output")}.mp4'
        video.write_videofile(output, fps=30, codec='libx264', audio_codec='aac', verbose=False, logger=None)
        
        print(f"\n✅ BGM 비디오 생성 완료!")
        print(f"🎵 BGM: 앰비언트 사운드")
        print(f"📹 길이: {video.duration}초")
        
        display(Video(output, embed=True, width=720))
        files.download(output)

---\n\n## 📝 사용법\n\n1. **Chrome Extension에서:**\n   - 이미지 수집/생성\n   - "Colab에서 편집" 클릭\n   - JSON 파일 다운로드\n\n2. **Colab에서:**\n   - 첫 번째 셀 실행\n   - JSON 파일 업로드\n   - 자동으로 비디오 생성 및 다운로드\n\n3. **커스터마이징:**\n   - 고급 설정 셀 사용\n   - BGM 추가 셀 사용\n\n---\n\n*Chrome Extension Video Generator v1.0*